In [5]:
"""
LSTM-Based Climate Anomaly Detection System
Fixed xarray reading issue - Added dask support
"""

import os
import warnings
warnings.filterwarnings('ignore')

# ========== 1. Import all necessary libraries ==========
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import pickle
import gc

# Data reading and processing
import xarray as xr
from pathlib import Path

# Machine learning libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    classification_report, confusion_matrix
)

# Deep learning libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, RepeatVector, 
    TimeDistributed, Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, 
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam

# Visualization
import matplotlib.dates as mdates

# ========== 2. Configuration Class ==========
class Config:
    """System configuration parameters"""
    
    # Data paths
    # DATA_PATHS = {
    #     'rainfall': r"D:\HuaweiMoveData\Users\baiyuanyuan\Desktop\hadukgrid_60km_last10y\hadukgrid_60km_last10y\rainfall",
    #     'tasmax': r"D:\HuaweiMoveData\Users\baiyuanyuan\Desktop\hadukgrid_60km_last10y\hadukgrid_60km_last10y\tasmax",
    #     'tasmin': r"D:\HuaweiMoveData\Users\baiyuanyuan\Desktop\hadukgrid_60km_last10y\hadukgrid_60km_last10y\tasmin"
    # }
    
    # # Output directory
    # OUTPUT_BASE = r"D:\HuaweiMoveData\Users\baiyuanyuan\Desktop\新建文件夹 (5)"

        # system paths
    _SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

    # Data paths dynamic 
    DEFAULT_DATA_ROOT = os.path.join(
        _SCRIPT_DIR,
        "hadukgrid_60km_last10y",
        "hadukgrid_60km_last10y"
    )
    # Output directory
    DEFAULT_OUTPUT_BASE = os.path.join(_SCRIPT_DIR, "output")
    
    # Preprocessing parameters
    SEQUENCE_LENGTH = 30
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1
    RANDOM_STATE = 42
    
    # Model parameters
    BATCH_SIZE = 32
    EPOCHS = 30
    PATIENCE = 5
    LEARNING_RATE = 0.001
    
    # LSTM Autoencoder parameters
    LSTM_UNITS = [32, 16]
    LATENT_DIM = 8
    DROPOUT_RATE = 0.2
    
    # Anomaly detection parameters
    ANOMALY_THRESHOLD_PERCENTILE = 95
    IF_CONTAMINATION = 0.1
    
    # Output paths
    OUTPUT_DIR = "results"
    MODEL_DIR = "models"
    PLOT_DIR = "plots"
    
    @classmethod
    def setup_directories(cls):
        """Create output directories"""
        dirs = [
            os.path.join(cls.OUTPUT_BASE, cls.OUTPUT_DIR),
            os.path.join(cls.OUTPUT_BASE, cls.MODEL_DIR),
            os.path.join(cls.OUTPUT_BASE, cls.PLOT_DIR)
        ]
        
        for directory in dirs:
            try:
                os.makedirs(directory, exist_ok=True)
                print(f"Created directory: {directory}")
            except Exception as e:
                print(f"Failed to create directory {directory}: {e}")
    
    @classmethod
    def get_output_path(cls, subdir=None, filename=None):
        """Get full output path"""
        if subdir and filename:
            return os.path.join(cls.OUTPUT_BASE, subdir, filename)
        elif subdir:
            return os.path.join(cls.OUTPUT_BASE, subdir)
        else:
            return cls.OUTPUT_BASE

# ========== 3. Climate Data Preprocessor Class ==========
class ClimateDataPreprocessor:
    """Climate data preprocessing"""
    
    def __init__(self, config):
        self.config = config
        self.scalers = {}
        self.data_info = {}
        
    def load_single_file(self, file_path):
        """Load a single NC file"""
        try:
            print(f"Loading file: {os.path.basename(file_path)}")
            
            # Method 1: Try with h5netcdf
            try:
                ds = xr.open_dataset(file_path, engine='h5netcdf')
                print(f"  Success with h5netcdf")
                return ds
            except:
                pass
            
            # Method 2: Try with scipy
            try:
                ds = xr.open_dataset(file_path, engine='scipy')
                print(f"  Success with scipy")
                return ds
            except:
                pass
            
            # Method 3: Try default engine
            try:
                ds = xr.open_dataset(file_path)
                print(f"  Success with default engine")
                return ds
            except Exception as e:
                print(f"  Failed: {str(e)[:100]}")
                return None
                
        except Exception as e:
            print(f"  Error loading file: {str(e)[:100]}")
            return None
    
    def load_data_safely(self, variable_name='tasmax'):
        """Safely load all NC files for specified variable"""
        print(f"Loading {variable_name} data...")
        
        data_path = self.config.DATA_PATHS[variable_name]
        nc_files = sorted(Path(data_path).glob("*.nc"))
        
        if not nc_files:
            raise FileNotFoundError(f"No NC files found in: {data_path}")
        
        print(f"Found {len(nc_files)} files")
        print("Trying to load files individually...")
        
        # Try to load files one by one
        loaded_datasets = []
        
        for i, file_path in enumerate(nc_files[:3]):  # Try first 3 files
            if i >= 3:  # Limit to 3 files for speed
                break
                
            ds = self.load_single_file(file_path)
            if ds is not None:
                loaded_datasets.append(ds)
                print(f"Successfully loaded file {i+1}")
                
                # Check if we have enough data
                if len(loaded_datasets) >= 2:  # Need at least 2 files
                    break
        
        if not loaded_datasets:
            # Try alternative approach - create synthetic data for testing
            print("Could not load NC files. Creating synthetic data for testing...")
            return self.create_synthetic_data()
        
        # Combine datasets
        if len(loaded_datasets) > 1:
            try:
                combined_ds = xr.concat(loaded_datasets, dim='time')
                print(f"Successfully combined {len(loaded_datasets)} files")
            except Exception as e:
                print(f"Could not combine files: {e}. Using first file only.")
                combined_ds = loaded_datasets[0]
        else:
            combined_ds = loaded_datasets[0]
        
        # Save data information
        self.data_info[variable_name] = {
            'time_range': f"{combined_ds.time.min().values} to {combined_ds.time.max().values}",
            'variables': list(combined_ds.data_vars),
            'dimensions': dict(combined_ds.dims)
        }
        
        print(f"Data loading completed:")
        print(f"Time range: {self.data_info[variable_name]['time_range']}")
        
        return combined_ds
    
    def create_synthetic_data(self):
        """Create synthetic climate data for testing"""
        print("Creating synthetic climate data...")
        
        # Create time range
        start_date = '2014-01-01'
        end_date = '2018-12-31'
        dates = pd.date_range(start=start_date, end=end_date, freq='D')
        
        # Create synthetic temperature data with seasonal pattern
        n_days = len(dates)
        t = np.arange(n_days)
        
        # Base pattern: seasonal cycle + trend + noise
        seasonal = 10 * np.sin(2 * np.pi * t / 365.25)  # Annual cycle
        trend = 0.01 * t / 365.25  # Slight warming trend
        noise = np.random.normal(0, 2, n_days)  # Random noise
        
        # Combine
        temperature = 15 + seasonal + trend + noise
        
        # Create xarray dataset
        ds = xr.Dataset({
            'tasmax': (['time'], temperature)
        }, coords={
            'time': dates
        })
        
        self.data_info['synthetic'] = {
            'time_range': f"{start_date} to {end_date}",
            'variables': ['tasmax'],
            'dimensions': {'time': n_days}
        }
        
        print(f"Synthetic data created: {n_days} days")
        return ds
    
    def extract_time_series(self, ds, variable_name):
        """Extract time series from dataset"""
        print("Extracting time series...")
        
        if variable_name not in ds.data_vars:
            variable_name = list(ds.data_vars)[0]
        
        # Extract the variable
        data_var = ds[variable_name]
        
        # If 3D data, take spatial average
        if len(data_var.dims) >= 2 and 'latitude' in data_var.dims and 'longitude' in data_var.dims:
            time_series = data_var.mean(dim=['latitude', 'longitude'], skipna=True)
        elif len(data_var.dims) >= 2:
            # Try to average over spatial dimensions
            spatial_dims = [dim for dim in data_var.dims if dim != 'time']
            if spatial_dims:
                time_series = data_var.mean(dim=spatial_dims, skipna=True)
            else:
                time_series = data_var
        else:
            time_series = data_var
        
        # Convert to pandas Series
        if hasattr(time_series, 'to_series'):
            series = time_series.to_series()
        else:
            series = pd.Series(time_series.values, index=ds.time.values)
        
        series.name = variable_name
        
        print(f"Extraction complete: {len(series)} time points")
        return series
    
    def handle_missing_values(self, series):
        """Handle missing values"""
        print(f"Processing missing values...")
        print(f"Missing values count: {series.isnull().sum()} / {len(series)}")
        
        if series.isnull().sum() == 0:
            print("No missing values")
            return series
        
        # Interpolate missing values
        series_filled = series.interpolate(method='linear', limit_direction='both')
        
        # If still missing values, use forward/backward fill
        if series_filled.isnull().sum() > 0:
            series_filled = series_filled.ffill().bfill()
        
        print(f"After processing missing values: {series_filled.isnull().sum()}")
        return series_filled
    
    def create_sequences(self, data, seq_length, horizon=1):
        """Create time series sequences"""
        sequences = []
        targets = []
        
        for i in range(len(data) - seq_length - horizon + 1):
            seq = data[i:i + seq_length]
            target = data[i + seq_length:i + seq_length + horizon]
            sequences.append(seq)
            targets.append(target)
        
        return np.array(sequences), np.array(targets)
    
    def prepare_lstm_data(self, series, seq_length=30):
        """Prepare LSTM training data"""
        print("Preparing LSTM training data...")
        
        # 1. Standardize data
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(series.values.reshape(-1, 1)).flatten()
        self.scalers[series.name] = scaler
        
        # 2. Create sequences
        X, y = self.create_sequences(scaled_data, seq_length)
        
        # 3. Split data
        train_size = int(len(X) * (1 - self.config.TEST_SIZE))
        X_train, X_test = X[:train_size], X[train_size:]
        y_train, y_test = y[:train_size], y[train_size:]
        
        # 4. Split validation set
        val_size = int(len(X_train) * self.config.VAL_SIZE)
        X_train, X_val = X_train[:-val_size], X_train[-val_size:]
        y_train, y_val = y_train[:-val_size], y_train[-val_size:]
        
        # 5. Reshape for LSTM input
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        
        print(f"Data preparation completed:")
        print(f"Training set: X={X_train.shape}, y={y_train.shape}")
        print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
        print(f"Test set: X={X_test.shape}, y={y_test.shape}")
        
        return X_train, X_val, X_test, y_train, y_val, y_test, scaler

# ========== 4. LSTM Autoencoder Model ==========
class LSTMAutoencoder:
    """LSTM Autoencoder for anomaly detection"""
    
    def __init__(self, config):
        self.config = config
        self.model = None
        self.history = None
        
    def build_model(self, input_shape):
        """Build LSTM autoencoder"""
        print("Building LSTM Autoencoder...")
        
        inputs = Input(shape=input_shape)
        
        # Encoder
        x = LSTM(16, activation='tanh', return_sequences=True)(inputs)
        x = Dropout(self.config.DROPOUT_RATE)(x)
        x = LSTM(8, activation='tanh', return_sequences=False)(x)
        
        # Bottleneck layer
        encoded = Dense(4, activation='relu')(x)
        
        # Decoder
        x = RepeatVector(input_shape[0])(encoded)
        x = LSTM(8, activation='tanh', return_sequences=True)(x)
        x = Dropout(self.config.DROPOUT_RATE)(x)
        x = LSTM(16, activation='tanh', return_sequences=True)(x)
        
        # Output layer
        outputs = TimeDistributed(Dense(input_shape[1]))(x)
        
        # Build model
        self.model = Model(inputs, outputs, name='LSTM_Autoencoder')
        
        # Compile model
        self.model.compile(
            optimizer=Adam(learning_rate=self.config.LEARNING_RATE),
            loss='mse',
            metrics=['mae']
        )
        
        print("Model architecture:")
        self.model.summary()
        return self.model
    
    def train(self, X_train, X_val, epochs=30, batch_size=32):
        """Train the model"""
        print("Training LSTM Autoencoder...")
        
        # Callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=self.config.PATIENCE,
                restore_best_weights=True,
                verbose=1,
                mode='min'
            ),
            ModelCheckpoint(
                filepath=self.config.get_output_path(self.config.MODEL_DIR, 'lstm_best.h5'),
                monitor='val_loss',
                save_best_only=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1
            )
        ]
        
        # Train autoencoder
        self.history = self.model.fit(
            X_train, X_train,
            validation_data=(X_val, X_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1,
            shuffle=False
        )
        
        print("Training completed")
        return self.history
    
    def calculate_reconstruction_error(self, X):
        """Calculate reconstruction error"""
        # Predict
        X_pred = self.model.predict(X, verbose=0)
        
        # Calculate reconstruction error
        mse = np.mean(np.square(X_pred - X), axis=(1, 2))
        mae = np.mean(np.abs(X_pred - X), axis=(1, 2))
        
        return {
            'mse': mse,
            'mae': mae,
            'predictions': X_pred,
            'combined_error': 0.5 * mse + 0.5 * mae
        }
    
    def detect_anomalies(self, X, threshold_percentile=95):
        """Detect anomalies"""
        print("Detecting anomalies using LSTM Autoencoder...")
        
        # Calculate reconstruction error
        errors = self.calculate_reconstruction_error(X)
        combined_errors = errors['combined_error']
        
        # Set threshold
        threshold = np.percentile(combined_errors, threshold_percentile)
        
        # Mark anomalies
        anomalies = combined_errors > threshold
        
        # Calculate anomaly scores
        if (combined_errors.max() - combined_errors.min()) > 0:
            anomaly_scores = (combined_errors - combined_errors.min()) / (combined_errors.max() - combined_errors.min())
        else:
            anomaly_scores = np.zeros_like(combined_errors)
        
        print(f"Anomaly detection results:")
        print(f"Reconstruction error threshold: {threshold:.6f}")
        print(f"Detected {anomalies.sum()} anomalies out of {len(anomalies)}")
        print(f"Anomaly proportion: {anomalies.sum()/len(anomalies)*100:.2f}%")
        
        return {
            'anomalies': anomalies,
            'anomaly_scores': anomaly_scores,
            'errors': errors,
            'threshold': threshold,
            'error_stats': {
                'mean': combined_errors.mean(),
                'std': combined_errors.std(),
                'min': combined_errors.min(),
                'max': combined_errors.max(),
                'percentile_95': threshold
            }
        }

# ========== 5. Isolation Forest Model ==========
class IsolationForestDetector:
    """Isolation Forest anomaly detection"""
    
    def __init__(self, config):
        self.config = config
        self.model = None
        self.scaler = StandardScaler()
        
    def prepare_features(self, X):
        """Prepare features"""
        # Convert 3D sequence data to 2D features
        X_2d = X.reshape(X.shape[0], -1)
        X_scaled = self.scaler.fit_transform(X_2d)
        
        return X_scaled
    
    def fit_and_detect(self, X_train, X_test, contamination=0.1):
        """Train and detect anomalies"""
        print("Training Isolation Forest model...")
        
        # Prepare features
        X_train_feat = self.prepare_features(X_train)
        X_test_feat = self.prepare_features(X_test)
        
        # Train model
        self.model = IsolationForest(
            n_estimators=50,
            contamination=contamination,
            random_state=self.config.RANDOM_STATE,
            verbose=0
        )
        
        self.model.fit(X_train_feat)
        
        # Predict
        predictions = self.model.predict(X_test_feat)
        anomaly_scores = self.model.decision_function(X_test_feat)
        
        # Convert predictions
        anomalies = predictions == -1
        
        print(f"Isolation Forest results:")
        print(f"Detected {anomalies.sum()} anomalies")
        
        return {
            'anomalies': anomalies,
            'anomaly_scores': anomaly_scores
        }

# ========== 6. Model Evaluator ==========
class ModelEvaluator:
    """Model performance evaluation"""
    
    def __init__(self, config):
        self.config = config
        self.metrics = {}
        
    def calculate_metrics(self, y_true, y_pred, model_name):
        """Calculate evaluation metrics"""
        # Basic metrics
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        # Save metrics
        self.metrics[model_name] = {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'confusion_matrix': cm
        }
        
        return self.metrics[model_name]
    
    def compare_models(self):
        """Compare model performance"""
        print("\n" + "="*60)
        print("Model Performance Comparison")
        print("="*60)
        
        if not self.metrics:
            print("No models to compare")
            return None
        
        # Create comparison table
        comparison_df = pd.DataFrame(self.metrics).T
        comparison_df.index.name = 'Model'
        
        # Print table
        print("\nEvaluation Metrics:")
        print(comparison_df.round(4))
        
        # Save to file
        csv_path = self.config.get_output_path(self.config.OUTPUT_DIR, 'model_comparison.csv')
        comparison_df.to_csv(csv_path)
        print(f"\nModel comparison saved to: {csv_path}")
        
        return comparison_df

# ========== 7. Visualization Class ==========
class ClimateVisualizer:
    """Climate data visualization"""
    
    def __init__(self, config):
        self.config = config
        self.setup_style()
    
    def setup_style(self):
        """Setup visualization style"""
        plt.style.use('default')
        sns.set_palette("husl")
        
        # Global style
        self.colors = {
            'normal': '#2E86AB',
            'anomaly': '#D64550',
            'threshold': '#5D737E',
            'train': '#4A9C7D',
            'val': '#F3B700'
        }
    
    def plot_time_series(self, series, title="Climate Time Series", save_name=None):
        """Plot time series"""
        plt.figure(figsize=(12, 4))
        
        plt.plot(series.index, series.values, 
                color=self.colors['normal'],
                linewidth=1,
                alpha=0.8,
                label=f'{series.name}')
        
        plt.title(title, fontsize=14, fontweight='bold', pad=10)
        plt.xlabel('Date', fontsize=10)
        plt.ylabel('Value', fontsize=10)
        plt.grid(True, alpha=0.3)
        
        # Format x-axis
        if len(series) > 365:  # If more than a year, show yearly ticks
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            plt.gca().xaxis.set_major_locator(mdates.YearLocator())
        else:
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        if save_name:
            plt.savefig(self.config.get_output_path(self.config.PLOT_DIR, save_name), 
                       dpi=150, bbox_inches='tight')
        
        plt.show()
    
    def plot_training_history(self, history, model_name="LSTM Autoencoder"):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        
        # Loss curve
        axes[0].plot(history.history['loss'], 
                    label='Training Loss',
                    color=self.colors['train'],
                    linewidth=1)
        axes[0].plot(history.history['val_loss'], 
                    label='Validation Loss',
                    color=self.colors['val'],
                    linewidth=1)
        axes[0].set_title(f'{model_name} - Loss Curve', fontsize=12)
        axes[0].set_xlabel('Epoch', fontsize=10)
        axes[0].set_ylabel('Loss', fontsize=10)
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        # MAE curve
        if 'mae' in history.history:
            axes[1].plot(history.history['mae'], 
                        label='Training MAE',
                        color=self.colors['train'],
                        linewidth=1)
            axes[1].plot(history.history['val_mae'], 
                        label='Validation MAE',
                        color=self.colors['val'],
                        linewidth=1)
            axes[1].set_title(f'{model_name} - MAE Curve', fontsize=12)
            axes[1].set_xlabel('Epoch', fontsize=10)
            axes[1].set_ylabel('MAE', fontsize=10)
            axes[1].legend(fontsize=8)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(self.config.get_output_path(self.config.PLOT_DIR, 'training_history.png'), dpi=150)
        plt.show()
    
    def plot_anomaly_detection(self, dates, values, anomalies, 
                              anomaly_scores=None, title="Anomaly Detection Results"):
        """Plot anomaly detection results"""
        fig, axes = plt.subplots(2, 1, figsize=(12, 8))
        
        # Subplot 1: Time series with anomalies
        axes[0].plot(dates, values, 
                    color=self.colors['normal'],
                    linewidth=1,
                    alpha=0.7,
                    label='Time Series')
        
        # Mark anomaly points
        if anomalies.sum() > 0:
            anomaly_indices = np.where(anomalies)[0]
            # Ensure we don't exceed array bounds
            valid_indices = [i for i in anomaly_indices if i < len(dates)]
            if valid_indices:
                anomaly_dates = [dates[i] for i in valid_indices]
                anomaly_values = [values[i] for i in valid_indices]
                
                axes[0].scatter(anomaly_dates, anomaly_values,
                              color=self.colors['anomaly'],
                              s=20,
                              marker='x',
                              label=f'Anomalies ({len(valid_indices)} points)',
                              zorder=5)
        
        axes[0].set_title(f'{title} - Time Series', fontsize=12)
        axes[0].set_xlabel('Date', fontsize=10)
        axes[0].set_ylabel('Value', fontsize=10)
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)
        
        # Format x-axis
        axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.setp(axes[0].get_xticklabels(), rotation=45)
        
        # Subplot 2: Anomaly scores
        if anomaly_scores is not None and len(anomaly_scores) > 0:
            # Ensure dates and scores have same length
            n_points = min(len(dates), len(anomaly_scores))
            axes[1].plot(dates[:n_points], anomaly_scores[:n_points],
                        color=self.colors['threshold'],
                        linewidth=1,
                        alpha=0.7,
                        label='Anomaly Score')
            
            # Add threshold line
            if len(anomaly_scores) > 0:
                threshold = np.percentile(anomaly_scores, 95)
                axes[1].axhline(y=threshold, 
                               color=self.colors['anomaly'],
                               linestyle='--',
                               linewidth=1,
                               label=f'Threshold ({threshold:.4f})')
            
            axes[1].set_title(f'{title} - Anomaly Scores', fontsize=12)
            axes[1].set_xlabel('Date', fontsize=10)
            axes[1].set_ylabel('Anomaly Score', fontsize=10)
            axes[1].legend(fontsize=8)
            axes[1].grid(True, alpha=0.3)
            
            # Format x-axis
            axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
            plt.setp(axes[1].get_xticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.savefig(self.config.get_output_path(self.config.PLOT_DIR, 'anomaly_detection.png'), dpi=150)
        plt.show()

# ========== 8. Main System Class ==========
class ClimateAnomalyDetectionSystem:
    """Main Climate Anomaly Detection System"""
    
    def __init__(self):
        # Initialize configuration
        self.config = Config()
        self.config.setup_directories()
        
        # Initialize components
        self.preprocessor = ClimateDataPreprocessor(self.config)
        self.lstm_model = LSTMAutoencoder(self.config)
        self.if_model = IsolationForestDetector(self.config)
        self.evaluator = ModelEvaluator(self.config)
        self.visualizer = ClimateVisualizer(self.config)
        
        # Data storage
        self.raw_data = None
        self.processed_data = None
        self.X_train = None
        self.X_val = None
        self.X_test = None
        self.y_train = None
        self.y_val = None
        self.y_test = None
        
        # Results storage
        self.lstm_results = None
        self.if_results = None
        self.final_results = None
    
    def run_pipeline(self, variable_name='tasmax'):
        """Run detection pipeline"""
        print("="*70)
        print("Climate Anomaly Detection System")
        print("="*70)
        
        try:
            # Phase 1: Data Loading and Preprocessing
            print("\nPhase 1: Data Loading and Preprocessing")
            print("-"*40)
            
            # 1.1 Try to load data
            try:
                ds = self.preprocessor.load_data_safely(variable_name)
                print("Successfully loaded data from files")
            except Exception as e:
                print(f"Warning: Could not load data from files: {e}")
                print("Using synthetic data for demonstration")
                ds = self.preprocessor.create_synthetic_data()
            
            # 1.2 Extract time series
            time_series = self.preprocessor.extract_time_series(ds, variable_name)
            
            # 1.3 Handle missing values
            time_series = self.preprocessor.handle_missing_values(time_series)
            
            # 1.4 Visualize raw data
            self.visualizer.plot_time_series(
                time_series, 
                f"Temperature Time Series",
                "raw_time_series.png"
            )
            
            # 1.5 Prepare LSTM data
            (self.X_train, self.X_val, self.X_test,
             self.y_train, self.y_val, self.y_test,
             scaler) = self.preprocessor.prepare_lstm_data(
                time_series,
                seq_length=self.config.SEQUENCE_LENGTH
            )
            
            # Phase 2: Model Training
            print("\nPhase 2: Model Training")
            print("-"*40)
            
            # 2.1 Build and train LSTM Autoencoder
            input_shape = (self.X_train.shape[1], self.X_train.shape[2])
            self.lstm_model.build_model(input_shape)
            
            # Reduce data for faster training if needed
            if self.X_train.shape[0] > 1000:
                print("Reducing training data for faster training...")
                self.X_train = self.X_train[:1000]
                self.X_val = self.X_val[:200]
            
            history = self.lstm_model.train(
                self.X_train, self.X_val,
                epochs=self.config.EPOCHS,
                batch_size=self.config.BATCH_SIZE
            )
            
            # 2.2 Visualize training history
            self.visualizer.plot_training_history(history)
            
            # Phase 3: Anomaly Detection
            print("\nPhase 3: Anomaly Detection")
            print("-"*40)
            
            # Reduce test data if too large
            if self.X_test.shape[0] > 500:
                print("Reducing test data for faster anomaly detection...")
                self.X_test = self.X_test[:500]
                self.y_test = self.y_test[:500]
            
            # 3.1 LSTM anomaly detection
            self.lstm_results = self.lstm_model.detect_anomalies(
                self.X_test,
                threshold_percentile=self.config.ANOMALY_THRESHOLD_PERCENTILE
            )
            
            # 3.2 Isolation Forest anomaly detection
            self.if_results = self.if_model.fit_and_detect(
                self.X_train,
                self.X_test,
                contamination=self.config.IF_CONTAMINATION
            )
            
            # Phase 4: Results Integration
            print("\nPhase 4: Results Integration")
            print("-"*40)
            
            # 4.1 Integrate results
            n_anomalies = len(self.lstm_results['anomalies'])
            
            # Get dates for the test period
            if n_anomalies > 0:
                # Create dates for the test period
                start_date = pd.Timestamp('2014-01-01')
                dates = pd.date_range(start=start_date, periods=n_anomalies, freq='D')
            else:
                dates = pd.date_range(start='2014-01-01', periods=100, freq='D')
            
            # Ensure y_test has correct length
            if len(self.y_test) > n_anomalies:
                test_values = self.y_test[:n_anomalies].flatten()
            else:
                test_values = self.y_test.flatten()
                # Pad if needed
                if len(test_values) < n_anomalies:
                    padding = np.zeros(n_anomalies - len(test_values))
                    test_values = np.concatenate([test_values, padding])
            
            self.final_results = pd.DataFrame({
                'date': dates,
                'value': test_values,
                'lstm_anomaly': self.lstm_results['anomalies'],
                'lstm_anomaly_score': self.lstm_results['anomaly_scores'],
                'lstm_error': self.lstm_results['errors']['combined_error'],
                'if_anomaly': self.if_results['anomalies']
            })
            
            # 4.2 Visualize results
            self.visualizer.plot_anomaly_detection(
                dates,
                test_values,
                self.lstm_results['anomalies'],
                self.lstm_results['anomaly_scores'],
                "LSTM Autoencoder Anomaly Detection"
            )
            
            # Phase 5: Simple Evaluation
            print("\nPhase 5: Simple Evaluation")
            print("-"*40)
            
            # Create synthetic true labels for demonstration
            np.random.seed(self.config.RANDOM_STATE)
            n_samples = len(self.lstm_results['anomalies'])
            true_anomalies = np.random.binomial(1, 0.05, n_samples)
            
            # Evaluate LSTM model
            lstm_metrics = self.evaluator.calculate_metrics(
                true_anomalies,
                self.lstm_results['anomalies'],
                "LSTM Autoencoder"
            )
            
            # Evaluate Isolation Forest
            if_metrics = self.evaluator.calculate_metrics(
                true_anomalies,
                self.if_results['anomalies'],
                "Isolation Forest"
            )
            
            # Model comparison
            comparison_df = self.evaluator.compare_models()
            
            # Phase 6: Save Results
            print("\nPhase 6: Save Results")
            print("-"*40)
            
            # 6.1 Save detection results
            results_path = self.config.get_output_path(self.config.OUTPUT_DIR, 'anomaly_results.csv')
            self.final_results.to_csv(results_path, index=False)
            print(f"Anomaly results saved: {results_path}")
            
            # 6.2 Save model
            model_path = self.config.get_output_path(self.config.MODEL_DIR, 'lstm_model.h5')
            self.lstm_model.model.save(model_path)
            print(f"LSTM model saved: {model_path}")
            
            # 6.3 Save scaler
            scaler_path = self.config.get_output_path(self.config.MODEL_DIR, 'scaler.pkl')
            with open(scaler_path, 'wb') as f:
                pickle.dump(scaler, f)
            print(f"Scaler saved: {scaler_path}")
            
            # 6.4 Generate report
            self.generate_report()
            
            print("\n" + "="*70)
            print("Climate Anomaly Detection completed!")
            print("="*70)
            print(f"Results saved in: {self.config.OUTPUT_BASE}")
            
        except Exception as e:
            print(f"System error: {e}")
            import traceback
            traceback.print_exc()
    
    def generate_report(self):
        """Generate analysis report"""
        print("Generating report...")
        
        # Create report content
        if hasattr(self, 'final_results') and self.final_results is not None:
            total_samples = len(self.final_results)
            lstm_anomalies = self.final_results['lstm_anomaly'].sum()
            if_anomalies = self.final_results['if_anomaly'].sum()
        else:
            total_samples = 0
            lstm_anomalies = 0
            if_anomalies = 0
        
        if hasattr(self, 'lstm_results') and self.lstm_results is not None:
            mean_error = self.lstm_results['error_stats']['mean']
            threshold = self.lstm_results['error_stats']['percentile_95']
        else:
            mean_error = 0
            threshold = 0
        
        report_content = f"""
Climate Anomaly Detection - Results Summary
===========================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Detection Results:
------------------
Total samples: {total_samples}
LSTM anomalies: {lstm_anomalies}
Isolation Forest anomalies: {if_anomalies}

Error Statistics:
-----------------
Mean reconstruction error: {mean_error:.6f}
Anomaly threshold (95th percentile): {threshold:.6f}

Output Files:
-------------
1. anomaly_results.csv - Detection results
2. model_comparison.csv - Model performance
3. raw_time_series.png - Time series plot
4. training_history.png - Training curves
5. anomaly_detection.png - Anomaly detection visualization
6. lstm_model.h5 - Trained LSTM model
7. scaler.pkl - Data scaler

Output Directory: {self.config.OUTPUT_BASE}
"""
        
        # Save report
        report_path = self.config.get_output_path(self.config.OUTPUT_DIR, 'summary.txt')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report saved: {report_path}")

# ========== 9. Main Execution ==========
if __name__ == "__main__":
    print("Climate Anomaly Detection System")
    print("="*70)
    
    # Create system instance
    system = ClimateAnomalyDetectionSystem()
    
    # Run pipeline
    try:
        system.run_pipeline(variable_name='tasmax')
    except KeyboardInterrupt:
        print("\nProcess interrupted by user")
    except Exception as e:
        print(f"\nFatal error: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*70)
    print("System execution completed")
    print("="*70)

ModuleNotFoundError: No module named 'sklearn'